# 03 — Compare Parameter Sweeps

Load saved sweep results from `data/sweeps/` and compare across instruments,
timeframes, or strategies — without re-running any backtests.

**Prerequisites:** Run a sweep in `02_backtest_ema_cross.ipynb` (or any
strategy notebook) at least once.  Each sweep auto-saves to Parquet.

**Sections:**
1. Load sweeps — all or filtered by strategy / instrument / interval
2. Summary table — best params and key stats per sweep
3. Heatmaps — side-by-side PnL heatmaps for visual comparison
4. Parameter stability — do profitable regions overlap across sweeps?
5. Deep dive — pick any sweep and explore its full stats

In [ ]:
# ── Cell 1: Imports ────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.backtesting import load_sweeps
from charts import plot_pnl_heatmap

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Imports OK")

In [ ]:
# ── Cell 2: Load all sweeps ────────────────────────────────────────
#
# Load everything, or filter:
#   load_sweeps(strategy="EMACross")
#   load_sweeps(instrument_id="BTC-USD-PERP.HYPERLIQUID")
#   load_sweeps(bar_interval="1h")

sweeps = load_sweeps()

for label, df in sweeps.items():
    swept_at = df["_swept_at"].iloc[0]
    data_start = df["_data_start"].iloc[0]
    data_end = df["_data_end"].iloc[0]
    print(f"  {label}  ({len(df)} combos, swept {swept_at[:10]}, data {data_start[:10]} → {data_end[:10]})")

In [ ]:
# ── Cell 3: Best params per sweep ──────────────────────────────────
#
# For each sweep, find the combo with the highest total PnL and show
# key stats.  This is the "which params win where?" table.

summary_rows = []

# Detect param columns: anything that isn't a metadata col (prefixed _)
# or a known stats col.  We grab them from the first sweep.
_STATS_COLS = {
    "total_pnl", "total_pnl_pct", "num_positions", "final_balance",
    "min_balance", "error",
}

for label, df in sweeps.items():
    param_cols = [
        c for c in df.columns
        if not c.startswith("_") and c not in _STATS_COLS
        and c in df.columns[:20]  # params are always the first columns after metadata
    ]

    valid = df[df["total_pnl"].notna()]
    if valid.empty:
        continue

    best = valid.loc[valid["total_pnl"].idxmax()]

    row = {
        "sweep": label,
        "data_range": f"{best['_data_start'][:10]} → {best['_data_end'][:10]}",
    }
    # Add best param values
    for pc in param_cols:
        row[f"best_{pc}"] = best.get(pc)

    row.update({
        "pnl": best["total_pnl"],
        "pnl_pct": best["total_pnl_pct"],
        "positions": int(best["num_positions"]),
    })

    # Optional stats — present if analyzer succeeded
    for stat_key, display_key in [
        ("Max Drawdown", "max_dd"),
        ("Sharpe Ratio (252 days)", "sharpe"),
        ("Profit Factor", "profit_factor"),
        ("Win Rate", "win_rate"),
    ]:
        if stat_key in best.index:
            row[display_key] = best[stat_key]

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("=== Best Parameters Per Sweep ===")
display(summary_df)

In [ ]:
# ── Cell 4: Side-by-side PnL heatmaps ─────────────────────────────
#
# Shows one heatmap per sweep.  Only works for 2-parameter sweeps
# (which is the common case — fast/slow, entry/exit, etc.).
#
# Customize row_col / col_col if your params aren't named fast/slow.

ROW_COL = "slow"   # y-axis parameter
COL_COL = "fast"   # x-axis parameter

for label, df in sweeps.items():
    if ROW_COL not in df.columns or COL_COL not in df.columns:
        print(f"Skipping {label} — missing '{ROW_COL}' or '{COL_COL}' columns")
        continue
    plot_pnl_heatmap(
        df, row_col=ROW_COL, col_col=COL_COL,
        row_label=f"{ROW_COL.title()} Period",
        col_label=f"{COL_COL.title()} Period",
        title=f"Total PnL — {label}",
    )

In [ ]:
# ── Cell 5: Parameter stability ───────────────────────────────────
#
# For each param combo that appears in ALL sweeps, average the PnL
# across sweeps.  Combos that are profitable everywhere are more
# robust than combos that only work on one timeframe.
#
# This only makes sense when comparing the same strategy across
# different instruments or timeframes.

if len(sweeps) < 2:
    print("Need 2+ sweeps to analyse parameter stability.")
    print("Run your backtest notebook for another instrument or timeframe,")
    print("then re-run this notebook.")
else:
    # Detect param columns from first sweep
    first_df = next(iter(sweeps.values()))
    param_cols = [
        c for c in first_df.columns
        if not c.startswith("_") and c not in _STATS_COLS
        and c in first_df.columns[:20]
    ]

    # Tag each sweep's rows with a label, then concat
    tagged = []
    for label, df in sweeps.items():
        chunk = df[param_cols + ["total_pnl", "total_pnl_pct"]].copy()
        chunk["_label"] = label
        tagged.append(chunk)

    combined = pd.concat(tagged, ignore_index=True)

    # Average PnL% across sweeps for each param combo
    stability = (
        combined
        .groupby(param_cols, as_index=False)
        .agg(
            avg_pnl_pct=("total_pnl_pct", "mean"),
            min_pnl_pct=("total_pnl_pct", "min"),
            max_pnl_pct=("total_pnl_pct", "max"),
            sweep_count=("total_pnl_pct", "count"),
            all_profitable=("total_pnl", lambda x: (x > 0).all()),
        )
        .sort_values("avg_pnl_pct", ascending=False)
    )

    # Show combos that appear in all sweeps and are profitable everywhere
    n_sweeps = len(sweeps)
    robust = stability[
        (stability["sweep_count"] == n_sweeps) & stability["all_profitable"]
    ]

    if robust.empty:
        print(f"No param combos are profitable across all {n_sweeps} sweeps.")
        print("\nTop 10 by average PnL% (may not be profitable everywhere):")
        display(stability.head(10))
    else:
        print(f"{len(robust)} combos profitable across all {n_sweeps} sweeps:")
        display(robust.head(20))

    # Heatmap of average PnL% if we have 2 param cols
    if len(param_cols) == 2 and len(stability) > 1:
        plot_pnl_heatmap(
            stability, row_col=param_cols[1], col_col=param_cols[0],
            value_col="avg_pnl_pct",
            row_label=f"{param_cols[1].title()} Period",
            col_label=f"{param_cols[0].title()} Period",
            title=f"Avg PnL% Across {n_sweeps} Sweeps (robust params)",
            fmt=".1f",
        )

In [ ]:
# ── Cell 6: Deep dive into one sweep ──────────────────────────────
#
# Pick a sweep by label and explore its full stats DataFrame.

# Show available labels
print("Available sweeps:")
for i, label in enumerate(sweeps.keys()):
    print(f"  [{i}] {label}")

# ── Change this to pick a different sweep ──
PICK = 0

pick_label = list(sweeps.keys())[PICK]
pick_df = sweeps[pick_label]
print(f"\nInspecting: {pick_label}")
print(f"Columns: {list(pick_df.columns)}")
print(f"\nTop 10 by PnL:")
display(
    pick_df
    .sort_values("total_pnl", ascending=False)
    .head(10)
    .filter(regex=r"^(?!_)")  # hide metadata columns for readability
)